# Robustez de las intervenciones en Gvial: 10 réplicas pareadas

**Pregunta:** ¿las intervenciones A (enlace nuevo de 4,2 km) y B (Carrera 68 y Carrera 50 a 50 km/h) mejoran de verdad la robustez de la red vial, o la mejora observada con 3 réplicas es ruido?

El índice R de Schneider et al. (2011) es la media de la componente gigante S durante un ataque adaptativo de 10 pasos, cada uno de 1% de los tramos. La intermediación de cada paso se estima con 300 nodos de origen, así que dos corridas con semillas distintas dan valores distintos de R aunque la red sea la misma. Cada réplica usa una semilla; la base y cada escenario se comparan con la misma semilla (réplicas pareadas), y la diferencia es concluyente si su intervalo de confianza del 95% excluye el cero.

## 1. Entorno

In [ ]:
import sys
import time
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "osmnx==2.1.1", "networkx==3.7"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/proyecto-redes-bogota")
    RUTAS = {"src": BASE / "src", "grafos": BASE / "data" / "processed", "salida": BASE / "resultados" / "gvial"}
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RUTAS = {"src": RAIZ / "src", "grafos": RAIZ / "data" / "drive_snapshot" / "processed",
             "salida": RAIZ / "data" / "salidas_locales" / "resultados" / "gvial"}
RUTAS["salida"].mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(RUTAS["src"]))

import pandas as pd

import adaptador_gvial as ag
import pipeline_redes as pr

## 2. Parámetros y escenarios

In [ ]:
PARAM = {
    "enlace_A": (261630424, 261040900),     # enlace de 4,2 km entre la macrozona centro-occidente y la sur
    "corredores_B": {"Avenida Carrera 68", "Avenida Carrera 50"},
    "vel_kph": 50,
    "factor": 1.2,
    "robustez": {"pasos": 10, "frac": 0.01, "k": 300},
    "offsets": [0, 100, 200, 300, 400, 500, 600, 700, 800, 900],   # 10 réplicas
    "confianza": 0.95,
}

_, G = ag.cargar_gvial(RUTAS["grafos"], "gvial_arterial", "Gvial arterial")
H_A, km_A = pr.agregar_enlace(G, *PARAM["enlace_A"], PARAM["vel_kph"], PARAM["factor"])
H_B = pr.acelerar(G, PARAM["corredores_B"], PARAM["vel_kph"])
H_AB, _ = pr.agregar_enlace(H_B, *PARAM["enlace_A"], PARAM["vel_kph"], PARAM["factor"])
ESCENARIOS = {"base": G, "A": H_A, "B": H_B, "A+B": H_AB}
print(f"Enlace A: {km_A:.2f} km")

## 3. Réplicas

Cada valor de R se guarda apenas se calcula. Si la corrida se interrumpe, al volver a ejecutar se retoma donde quedó. Cada R toma alrededor de 1,5 minutos en un equipo local; el total son 40.

In [ ]:
ruta_reps = RUTAS["salida"] / "robustez_replicas.csv"
reps = pd.read_csv(ruta_reps) if ruta_reps.exists() else pd.DataFrame(columns=["escenario", "offset", "R"])
hechas = set(zip(reps["escenario"], reps["offset"]))

for offset in PARAM["offsets"]:
    for nombre, H in ESCENARIOS.items():
        if (nombre, offset) in hechas:
            continue
        t0 = time.time()
        R, _ = pr.robustez(H, offset=offset, **PARAM["robustez"])
        reps = pd.concat([reps, pd.DataFrame([{"escenario": nombre, "offset": offset, "R": R}])], ignore_index=True)
        reps.to_csv(ruta_reps, index=False)
        print(f"offset {offset:3d} | {nombre:4s} | R = {R:.4f} | {time.time() - t0:.0f} s")

tabla = reps.pivot(index="offset", columns="escenario", values="R")[list(ESCENARIOS)]
tabla.round(4)

## 4. ¿La mejora es real?

In [ ]:
resultado = pd.DataFrame({
    nombre: pr.comparar_replicas(tabla["base"], tabla[nombre], PARAM["confianza"])
    for nombre in ESCENARIOS if nombre != "base"
}).T
resultado.to_csv(RUTAS["salida"] / "robustez_intervenciones.csv")
resultado

### Consistencia con `01_gvial`

Las réplicas con offset 0, 100 y 200 son las mismas que calcula `01_gvial`; su media debe coincidir con `R_media` de `escenarios_final.csv`.

In [ ]:
ruta_final = RUTAS["salida"] / "escenarios_final.csv"
if ruta_final.exists():
    final = pd.read_csv(ruta_final).set_index("escenario")["R_media"]
    tres = tabla.loc[[0, 100, 200]].mean()
    print(pd.DataFrame({"01_gvial": final, "primeras 3 réplicas": tres}).round(6))
else:
    print("No hay escenarios_final.csv: correr 01_gvial primero para esta verificación")